# UNDERWOOD_J_LAB2
## Conformed Date & Time Dimensions and Slowly Changing Dimensions

This notebook extends the Lab 1 tables in `SNOWBEARAIR_DB.MODELED`. It keeps the existing Lab 1 column names, including `ACCOUNT_BILLING_COUNT`, which stores billing country. The calendar covers 2015–2030, fiscal years follow January–December, and business hours are 9:00 AM–4:59 PM.

Run all cells for the initial rebuild. For later daily updates, run the account and product MERGE cells without rerunning the DDL, which would reset their history. Account history uses daily snapshots; the initial source profile is assumed valid from its creation date.

---
## Setup
Run this cell first before anything else.

In [ ]:
%%sql -r dataframe_1
USE DATABASE SNOWBEARAIR_DB;
USE SCHEMA MODELED;
USE WAREHOUSE STUDENT_WH;

---
# Part 1: Design — Create the New and Updated Tables

**Build order:**
1. `dim_date` (no dependencies)
2. `dim_time` (no dependencies)
3. `dim_account` — rebuild with Type 2 columns
4. `dim_product` — rebuild with Type 3 column
5. `fact_opportunity_line_item` — rebuild with date/time foreign keys

---
### Task 1: Create dim_date

Create `dim_date_underwood_j`

**Requirements:**
- `date_key` as INTEGER — formatted as `YYYYMMDD` (e.g., `20240315` for March 15 2024). **Do not use AUTOINCREMENT** — the date itself is the key.
- One row per calendar date, covering at least 2015–2025
- At least 20 descriptive attributes (see the list in `lab2.md`)
- **SCD Type 0** — rows are never updated once inserted

**Tip:** dim_date is populated by *generating* rows, not by querying Salesforce.

In [ ]:
-- Task 1: Create dim_date_underwood_j
CREATE OR REPLACE TABLE dim_date_underwood_j (
    date_key              INTEGER PRIMARY KEY,
    calendar_date         DATE NOT NULL,
    day_number_in_month   INTEGER,
    day_name              VARCHAR(9),
    day_of_week_number    INTEGER,
    week_number           INTEGER,
    month_number          INTEGER,
    month_name            VARCHAR(9),
    quarter               INTEGER,
    year                  INTEGER,
    fiscal_month          INTEGER,
    fiscal_quarter        INTEGER,
    fiscal_year           INTEGER,
    is_weekend_flag       BOOLEAN,
    is_weekday_flag       BOOLEAN,
    month_start_flag      BOOLEAN,
    month_end_flag        BOOLEAN,
    quarter_start_flag    BOOLEAN,
    quarter_end_flag      BOOLEAN,
    year_start_flag       BOOLEAN,
    year_end_flag         BOOLEAN,
    day_of_year           INTEGER,
    iso_week_year         INTEGER
);

---
### Task 2: Create dim_time

Create `dim_time_underwood_j`

**Requirements:**
- `time_key` as INTEGER — formatted as `HHMM` (e.g., `1430` for 2:30 PM)
- One row per minute of the day (1,440 rows total)
- Required attributes: `hour_value`, `minute_value`, `second_value`, `am_pm_indicator`, `business_hours_flag`

In [ ]:
-- Task 2: Create dim_time_underwood_j
CREATE OR REPLACE TABLE dim_time_underwood_j (
    time_key              INTEGER PRIMARY KEY,
    hour_value            INTEGER,
    minute_value          INTEGER,
    second_value          INTEGER,
    am_pm_indicator       VARCHAR(2),
    business_hours_flag   BOOLEAN
);

---
### Task 3: Upgrade dim_account to SCD Type 2

Rebuild `dim_account_underwood_j` with full historical tracking.

**Add these three administrative columns to your Lab 1 structure:**
- `effective_start_date DATE` — first date this row's profile was valid
- `effective_end_date DATE` — last date valid; use `'9999-12-31'` for the current row (avoid NULLs)
- `current_flag BOOLEAN` — `TRUE` for the one active row per account

Use `CREATE OR REPLACE TABLE` — do not ALTER the Lab 1 table.

In [ ]:
-- Task 3: Preserve the Lab 1 attributes and add Type 2 history
CREATE OR REPLACE TABLE dim_account_underwood_j (
    account_key                 INTEGER AUTOINCREMENT PRIMARY KEY,
    account_id                  VARCHAR(255) NOT NULL,
    account_name                VARCHAR(255),
    account_type                VARCHAR(50),
    account_industry            VARCHAR(100),
    account_billing_state       VARCHAR(100),
    account_billing_count       VARCHAR(100),
    account_annual_revenue      NUMBER(18,2),
    account_number_of_employees INTEGER,
    account_created_date        DATE,
    effective_start_date        DATE NOT NULL,
    effective_end_date          DATE NOT NULL DEFAULT '9999-12-31'::DATE,
    current_flag                BOOLEAN NOT NULL DEFAULT TRUE
);

---
### Task 4: Upgrade dim_product to SCD Type 3

Rebuild `dim_product_underwood_j` to track current and prior product family.

**Add one column:**
- `prior_product_family_name VARCHAR(100)` — the family before the most recent reclassification

All other attributes continue using Type 1 behavior (overwrite on change).

In [ ]:
-- Task 4: Keep current product_family and add the prior family
CREATE OR REPLACE TABLE dim_product_underwood_j (
    product_key               INTEGER AUTOINCREMENT PRIMARY KEY,
    product_id                VARCHAR(255) NOT NULL,
    product_name              VARCHAR(255),
    product_code              VARCHAR(255),
    product_family            VARCHAR(255),
    product_is_active         BOOLEAN,
    prior_product_family_name VARCHAR(100)
);

---
### Task 5: Refactor the Fact Table

Rebuild `fact_opportunity_line_item_underwood_j` with role-playing date keys.

**Requirements:**
- Remove any raw DATE/TIMESTAMP columns
- Add `created_date_key` and `close_date_key` — both reference `dim_date_underwood_j` (role-playing)
- Add `created_time_key` — references `dim_time_underwood_j`
- Keep all existing FKs and measures from Lab 1

**Role-playing reminder:** You are NOT creating two separate date tables. Both date keys point to the same `dim_date_underwood_j` table — just with different column names.

In [ ]:
-- Task 5: One row per opportunity line item, with date/time keys
CREATE OR REPLACE TABLE fact_opportunity_line_item_underwood_j (
    line_item_key         INTEGER AUTOINCREMENT PRIMARY KEY,
    product_key           INTEGER NOT NULL,
    account_key           INTEGER NOT NULL,
    opportunity_key       INTEGER NOT NULL,
    created_date_key      INTEGER NOT NULL,
    close_date_key        INTEGER NOT NULL,
    created_time_key      INTEGER NOT NULL,
    line_item_quantity    NUMBER(18,2),
    line_item_unit_price  NUMBER(18,2),
    line_item_total_price NUMBER(18,2),
    FOREIGN KEY (product_key) REFERENCES dim_product_underwood_j(product_key),
    FOREIGN KEY (account_key) REFERENCES dim_account_underwood_j(account_key),
    FOREIGN KEY (opportunity_key) REFERENCES dim_opportunity_underwood_j(opportunity_key),
    FOREIGN KEY (created_date_key) REFERENCES dim_date_underwood_j(date_key),
    FOREIGN KEY (close_date_key) REFERENCES dim_date_underwood_j(date_key),
    FOREIGN KEY (created_time_key) REFERENCES dim_time_underwood_j(time_key)
);

---
# Part 2: Load Data

**Load order:** date → time → account → product → fact

`dim_date` and `dim_time` are generated from scratch — not queried from Salesforce.
`dim_account` and `dim_product` are reloaded from `SNOWBEARAIR_DB.PUBLIC` with new columns.
The fact table must be reloaded with joins to the date dimension to resolve date keys.

---
### Load dim_date

Generate rows using Snowflake's `GENERATOR` function — do **not** query from Salesforce.

See `dim_date_intro.md` for the `GENERATOR` + `SEQ4()` + `DATEADD` pattern and tips for calculating each attribute.

In [ ]:
-- 5,844 consecutive dates: January 1, 2015 through December 31, 2030
-- ROW_NUMBER avoids possible gaps in SEQ4(). Fiscal year = calendar year.
INSERT INTO dim_date_underwood_j (
    date_key, calendar_date, day_number_in_month, day_name,
    day_of_week_number, week_number, month_number, month_name,
    quarter, year, fiscal_month, fiscal_quarter, fiscal_year,
    is_weekend_flag, is_weekday_flag, month_start_flag, month_end_flag,
    quarter_start_flag, quarter_end_flag, year_start_flag, year_end_flag,
    day_of_year, iso_week_year
)
WITH date_spine AS (
    SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,
                   '2015-01-01'::DATE)::DATE AS d
    FROM TABLE(GENERATOR(ROWCOUNT => 5844))
)
SELECT
    TO_NUMBER(TO_CHAR(d, 'YYYYMMDD')),
    d,
    DAY(d),
    DECODE(DAYOFWEEKISO(d), 1, 'Monday', 2, 'Tuesday', 3, 'Wednesday',
           4, 'Thursday', 5, 'Friday', 6, 'Saturday', 7, 'Sunday'),
    MOD(DAYOFWEEKISO(d), 7) + 1,
    WEEKISO(d),
    MONTH(d),
    DECODE(MONTH(d), 1, 'January', 2, 'February', 3, 'March',
           4, 'April', 5, 'May', 6, 'June', 7, 'July', 8, 'August',
           9, 'September', 10, 'October', 11, 'November', 12, 'December'),
    QUARTER(d),
    YEAR(d),
    MONTH(d),
    QUARTER(d),
    YEAR(d),
    DAYOFWEEKISO(d) IN (6, 7),
    DAYOFWEEKISO(d) BETWEEN 1 AND 5,
    d = DATE_TRUNC('month', d),
    d = LAST_DAY(d, 'month'),
    d = DATE_TRUNC('quarter', d),
    d = LAST_DAY(d, 'quarter'),
    d = DATE_TRUNC('year', d),
    d = LAST_DAY(d, 'year'),
    DAYOFYEAR(d),
    YEAROFWEEKISO(d)
FROM date_spine;

---
### Load dim_time

Generate all 1,440 rows (one per minute of the day) using `GENERATOR`.
Apply the same GENERATOR pattern from `dim_date_intro.md` — adapt the math to produce hour and minute values instead of calendar dates.

In [ ]:
-- One row per minute; seconds are zero at this grain
INSERT INTO dim_time_underwood_j (
    time_key, hour_value, minute_value, second_value,
    am_pm_indicator, business_hours_flag
)
WITH minutes AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS minute_of_day
    FROM TABLE(GENERATOR(ROWCOUNT => 1440))
), clock_parts AS (
    SELECT FLOOR(minute_of_day / 60)::INTEGER AS h,
           MOD(minute_of_day, 60) AS m
    FROM minutes
)
SELECT h * 100 + m, h, m, 0,
       IFF(h < 12, 'AM', 'PM'),
       h >= 9 AND h < 17
FROM clock_parts;

---
### Load dim_account (Type 2 — MERGE)

Use `MERGE INTO` to load `dim_account_underwood_j`.

- `WHEN NOT MATCHED` — insert new accounts with `current_flag = TRUE`, `effective_end_date = '9999-12-31'`
- `WHEN MATCHED` — for changed accounts, use the two-step expire + insert pattern described in `merge_intro.md`

> Match on the natural key (`account_id` ↔ source `id`).

In [ ]:
-- Type 2: expire changed profiles, then insert new current versions.
-- Keep both steps in one transaction. Run once per daily snapshot.
BEGIN TRANSACTION;

MERGE INTO dim_account_underwood_j AS target
USING SNOWBEARAIR_DB.PUBLIC.account AS source
    ON target.account_id = source.id AND target.current_flag = TRUE
WHEN MATCHED AND (
       target.account_name IS DISTINCT FROM source.name
    OR target.account_type IS DISTINCT FROM source.type
    OR target.account_industry IS DISTINCT FROM source.industry
    OR target.account_billing_state IS DISTINCT FROM source.billing_state
    OR target.account_billing_count IS DISTINCT FROM source.billing_country
    OR target.account_annual_revenue IS DISTINCT FROM source.annual_revenue
    OR target.account_number_of_employees IS DISTINCT FROM source.number_of_employees
    OR target.account_created_date IS DISTINCT FROM source.created_date
) THEN UPDATE SET
    target.effective_end_date = DATEADD(day, -1, CURRENT_DATE()),
    target.current_flag = FALSE;

MERGE INTO dim_account_underwood_j AS target
USING (
    SELECT s.*,
           IFF(h.account_id IS NULL, COALESCE(s.created_date, CURRENT_DATE()),
               CURRENT_DATE()) AS version_start_date
    FROM SNOWBEARAIR_DB.PUBLIC.account s
    LEFT JOIN (SELECT DISTINCT account_id FROM dim_account_underwood_j) h
        ON h.account_id = s.id
) AS source
    ON target.account_id = source.id AND target.current_flag = TRUE
WHEN NOT MATCHED THEN INSERT (
    account_id, account_name, account_type, account_industry,
    account_billing_state, account_billing_count, account_annual_revenue,
    account_number_of_employees, account_created_date,
    effective_start_date, effective_end_date, current_flag
) VALUES (
    source.id, source.name, source.type, source.industry,
    source.billing_state, source.billing_country, source.annual_revenue,
    source.number_of_employees, source.created_date,
    source.version_start_date, '9999-12-31'::DATE, TRUE
);

COMMIT;

---
### Load dim_product (Type 3 — MERGE)

Use `MERGE INTO` to load `dim_product_underwood_j`.

- `WHEN NOT MATCHED` — insert new products with `prior_product_family_name = NULL`
- `WHEN MATCHED` — update attributes; when the family changes, shift current → prior before writing the new value

> Match on the natural key (`product_id` ↔ source `id`). See `merge_intro.md` for the Type 3 update pattern.

In [ ]:
-- Type 3 for family; Type 1 for name, code and active flag
MERGE INTO dim_product_underwood_j AS target
USING SNOWBEARAIR_DB.PUBLIC.product AS source
    ON target.product_id = source.id
WHEN MATCHED AND (
       target.product_name IS DISTINCT FROM source.name
    OR target.product_code IS DISTINCT FROM source.product_code
    OR target.product_family IS DISTINCT FROM source.family
    OR target.product_is_active IS DISTINCT FROM source.is_active
) THEN UPDATE SET
    target.prior_product_family_name = IFF(
        target.product_family IS DISTINCT FROM source.family,
        target.product_family, target.prior_product_family_name),
    target.product_family = source.family,
    target.product_name = source.name,
    target.product_code = source.product_code,
    target.product_is_active = source.is_active
WHEN NOT MATCHED THEN INSERT (
    product_id, product_name, product_code, product_family,
    product_is_active, prior_product_family_name
) VALUES (
    source.id, source.name, source.product_code, source.family,
    source.is_active, NULL
);

---
### Reload fact_opportunity_line_item

Reload the fact table with the new date/time key columns.

**Date key resolution:** Convert the source date to YYYYMMDD integer format and join to `dim_date_underwood_j` — same pattern shown in `dim_date_intro.md`.

**Role-playing joins:** Join `dim_date_underwood_j` twice with different aliases — once for `created_date_key`, once for `close_date_key`.

**Type 2 join note:** Add `AND a.current_flag = TRUE` when joining `dim_account_underwood_j` to get the current account profile.

In [ ]:
-- Initial reload: join current accounts as directed by the lab template.
-- Run this initial fact load once after recreating the empty fact table.
INSERT INTO fact_opportunity_line_item_underwood_j (
    product_key, account_key, opportunity_key,
    created_date_key, close_date_key, created_time_key,
    line_item_quantity, line_item_unit_price, line_item_total_price
)
SELECT
    p.product_key, a.account_key, op.opportunity_key,
    created.date_key, closed.date_key, t.time_key,
    oli.quantity, oli.unit_price, oli.total_price
FROM SNOWBEARAIR_DB.PUBLIC.opportunity_line_item oli
JOIN SNOWBEARAIR_DB.PUBLIC.opportunity o ON o.id = oli.opportunity_id
JOIN dim_product_underwood_j p ON p.product_id = oli.product_id
JOIN dim_account_underwood_j a
    ON a.account_id = o.account_id AND a.current_flag = TRUE
JOIN dim_opportunity_underwood_j op ON op.opportunity_id = o.id
JOIN dim_date_underwood_j created
    ON created.date_key = TO_NUMBER(TO_CHAR(o.created_date, 'YYYYMMDD'))
JOIN dim_date_underwood_j closed
    ON closed.date_key = TO_NUMBER(TO_CHAR(o.close_date, 'YYYYMMDD'))
JOIN dim_time_underwood_j t
    ON t.time_key = HOUR(o.created_date) * 100 + MINUTE(o.created_date);

---
## Verify Your Work

Run these queries to confirm correctness before submitting.

In [ ]:
%%sql -r dataframe_12
-- Expected: date 5844, time 1440, account 3, product 10, opportunity 3, fact 4
SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM dim_date_underwood_j
UNION ALL SELECT 'dim_time', COUNT(*) FROM dim_time_underwood_j
UNION ALL SELECT 'dim_account', COUNT(*) FROM dim_account_underwood_j
UNION ALL SELECT 'dim_product', COUNT(*) FROM dim_product_underwood_j
UNION ALL SELECT 'dim_opportunity', COUNT(*) FROM dim_opportunity_underwood_j
UNION ALL SELECT 'fact_opportunity_line_item', COUNT(*) FROM fact_opportunity_line_item_underwood_j;

-- Date range and unique keys
SELECT MIN(calendar_date) AS first_date, MAX(calendar_date) AS last_date,
       COUNT(*) AS dates, COUNT(DISTINCT date_key) AS unique_keys
FROM dim_date_underwood_j;

-- Exactly one current version per account; expect zero rows
SELECT account_id, COUNT_IF(current_flag) AS current_rows
FROM dim_account_underwood_j
GROUP BY account_id
HAVING current_rows != 1;

-- Every fact must resolve all six dimension relationships; expect 4 rows
SELECT COUNT(*) AS fully_joined_fact_rows,
       SUM(f.line_item_quantity) AS quantity,
       SUM(f.line_item_total_price) AS revenue
FROM fact_opportunity_line_item_underwood_j f
JOIN dim_product_underwood_j p ON p.product_key = f.product_key
JOIN dim_account_underwood_j a ON a.account_key = f.account_key
JOIN dim_opportunity_underwood_j o ON o.opportunity_key = f.opportunity_key
JOIN dim_date_underwood_j created ON created.date_key = f.created_date_key
JOIN dim_date_underwood_j closed ON closed.date_key = f.close_date_key
JOIN dim_time_underwood_j t ON t.time_key = f.created_time_key;

-- Source totals should match the modeled totals above
SELECT COUNT(*) AS source_rows, SUM(quantity) AS quantity, SUM(total_price) AS revenue
FROM SNOWBEARAIR_DB.PUBLIC.opportunity_line_item;

-- Both date roles in one query
SELECT created.year AS created_year, created.quarter AS created_quarter,
       closed.year AS close_year, closed.quarter AS close_quarter,
       SUM(f.line_item_total_price) AS revenue
FROM fact_opportunity_line_item_underwood_j f
JOIN dim_date_underwood_j created ON created.date_key = f.created_date_key
JOIN dim_date_underwood_j closed ON closed.date_key = f.close_date_key
GROUP BY created.year, created.quarter, closed.year, closed.quarter
ORDER BY created_year, created_quarter, close_year, close_quarter;

-- No raw date/timestamp columns in the fact table; expect zero rows
SELECT column_name, data_type
FROM SNOWBEARAIR_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MODELED'
  AND table_name = 'FACT_OPPORTUNITY_LINE_ITEM_UNDERWOOD_J'
  AND (data_type = 'DATE' OR data_type LIKE 'TIMESTAMP%');

-- Final summary for the notebook result grid and HTML export
WITH checks AS (
    SELECT 1 AS check_order, 'Date rows' AS check_name, COUNT(*) AS actual, 5844 AS expected FROM dim_date_underwood_j
    UNION ALL SELECT 2, 'Time rows', COUNT(*), 1440 FROM dim_time_underwood_j
    UNION ALL SELECT 3, 'Account rows', COUNT(*), 3 FROM dim_account_underwood_j
    UNION ALL SELECT 4, 'Product rows', COUNT(*), 10 FROM dim_product_underwood_j
    UNION ALL SELECT 5, 'Opportunity rows', COUNT(*), 3 FROM dim_opportunity_underwood_j
    UNION ALL SELECT 6, 'Fact rows', COUNT(*), 4 FROM fact_opportunity_line_item_underwood_j
    UNION ALL SELECT 7, 'Total quantity', SUM(line_item_quantity), 22 FROM fact_opportunity_line_item_underwood_j
    UNION ALL SELECT 8, 'Total revenue', SUM(line_item_total_price), 360000 FROM fact_opportunity_line_item_underwood_j
)
SELECT check_name, actual, expected, IFF(actual = expected, 'PASS', 'FAIL') AS result
FROM checks
ORDER BY check_order;

## Written explanation

**Notebook name:** `UNDERWOOD_J_LAB2`  
**SQL context:** `SNOWBEARAIR_DB.MODELED`

**Account Type 2:** Type 2 keeps old account profiles when details change. Type 1 would overwrite them and lose the history needed to explain past sales.

**Product Type 3:** Type 3 stores the current and previous product family. That is enough for a before-and-after comparison; Type 1 loses the previous family, while Type 2 stores more history than this task needs.

**Role-playing dimensions:** One date table serves different purposes. I joined `dim_date_underwood_j` twice, using `created_date_key` for creation dates and `close_date_key` for closing dates.

## Results

The model contains 5,844 dates, 1,440 time entries, 3 accounts, 10 products, 3 opportunities, and 4 sales records. All sales link to their dimensions, with total quantity 22 and revenue $360,000 matching the source. Rerunning the unchanged account and product loads creates no duplicates. Account history and product family changes also passed 16 checks using temporary copies.